# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their columns, and their `@id`s.

Let's inspect which record sets and fields are present in the dataset:

In [ ]:
# List all record sets and their fields, referencing everything by @id
print("Available record sets and their fields (referenced by @id):\n")
record_sets = dataset.record_sets
if not record_sets:
    print("No explicit record sets found in the schema; the main tabular asset will be used.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs['@id'] if '@id' in rs else rs.get('name', '<unknown>')}\n  Fields:")
        for field in rs.get('field', []):
            if isinstance(field, dict):
                print(f"    - Field: {field.get('@id', '<no_id>')}, column: {field.get('column', '<no_column>')}")
            else:
                print(f"    - Field: {field}")


**Note:** If the dataset defines only a single main file (no explicit record sets), we use that as a single tabular record set for loading records.

To see some records, let's query them by providing the `record_set` argument. If there are no defined record sets, we'll load the default one via an empty argument.

In [ ]:
# Try to list the first 3 records for demonstration
print("First 3 records from the main record set:")
# Try to get the default/main record set @id (if present)
if record_sets:
    # If there are defined record sets, pick the first one's @id
    main_record_set_id = record_sets[0].get('@id', None)
    for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
        if i >= 3:
            break
        print(rec)
else:
    # No record set defined, use default
    for i, rec in enumerate(dataset.records()):
        if i >= 3:
            break
        print(rec)


## 3. Data Extraction
Load data from the available record set into a DataFrame for analysis. All references are made using `@id`s as per best practices.

**Note:** Since this dataset does not explicitly define multiple record sets, we will work with a single DataFrame loaded from the main (default) reference.

In [ ]:
# If there are explicit record sets, collect them by their @id; otherwise use [None]
if record_sets:
    record_set_ids = [rs.get('@id', None) for rs in record_sets]
else:
    # The default for single-table datasets
    record_set_ids = [None]
        
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

main_df = dataframes[record_set_ids[0]]
print("Available columns (fields referenced by @id):")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's:
- Select the GAD-7 total score field by its `@id` (`gad7_total_score` if schema preserves such field names)
- Filter for records where GAD-7 total score > 10 (cutoff for moderate/severe anxiety)
- Normalize the GAD-7 total score
- Group by the `@id` for 'level_of_education`, if available

**Replace the field names below with their actual @ids as found in the DataFrame** for most reliable operation. We'll show an example using plausible field ids from a typical mental-health survey.

In [ ]:
# Assign the (plausible) @ids for the fields based on schema/column headers printed above
# Please adapt if needed to actual @ids listed in main_df.columns

# Typical field @ids in such datasets:
gad7_field_id_candidates = [c for c in main_df.columns if 'gad7' in c.lower() and ('total' in c.lower() or c.lower().endswith('score'))]
if gad7_field_id_candidates:
    numeric_field = gad7_field_id_candidates[0]  # Pick the best match
else:
    # fallback to any numeric column
    numeric_field = main_df.select_dtypes('number').columns[0]

# Choose plausible field @id for grouping (e.g., level_of_education)
group_field_candidates = [c for c in main_df.columns if 'level' in c.lower() and 'educ' in c.lower()]
group_field = group_field_candidates[0] if group_field_candidates else None

threshold = 10
filtered_df = main_df[main_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field (e.g., GAD-7 total score)
sns.histplot(main_df[numeric_field].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

if group_field is not None:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=main_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and perform basic analysis of the FAIR² Kilifi County Mental Health Survey Dataset using the `mlcroissant` library.

- We loaded the dataset using its Croissant schema URL.
- We reviewed available fields/columns (referenced strictly by their `@id`s), and loaded the data into pandas DataFrames.
- We performed basic exploratory steps: filtering for high anxiety respondents, normalizing scores, and grouping by education level (if available).
- We visualized the distribution of GAD-7 scores and their relation to education level.

For further research, you may wish to look for other assessment fields (e.g., PHQ-9, PSQ), stratify by demographic variables, or map additional `@id`s to standard ontologies.